# CONFIGURAR E CARREGAR DE DATASET

In [287]:
# ============================================================
# Importa as bibliotecas do Python
# ============================================================

# Importar o pandas
import pandas as pd
import numpy as np

# Importar o LabelEncoder da biblioteca scikit-learn
from sklearn.preprocessing import LabelEncoder

# Importar a função de divisão de treino e teste
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Modelos
from sklearn.ensemble import RandomForestClassifier
from xgboost          import XGBClassifier
from sklearn.metrics  import (accuracy_score, precision_score,
                               recall_score, f1_score, make_scorer,
                               classification_report,  confusion_matrix, roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV


import warnings
warnings.filterwarnings('ignore')





In [288]:
# ============================================================
# Carregar o dataset a partir do arquivo Excel
# ============================================================

# URL do arquivo no GitHub
url_dataset = "https://raw.githubusercontent.com/gilbertoag2007/machine-learning-demandas-ti/main/DEMANDAS_DOWNSTREAM_V4.xlsx"

# Cria um dataframe com o conteúdo do arquivo carregado
colunas_desejadas = ["ID_DEMANDA", "SISTEMA", "NOME_EQUIPE", "TIPO_DEMANDA", "NUM_DEMANDA", "ANO_DEMANDA", "SITUACAO_DEMANDA","DAT_INICIAL_CLASSIFICACAO", "DAT_PREVISTA_INI_DEMANDA","DAT_PREVISTA_INI_REQUISITOS", "DAT_REAL_INI_REQ", "DAT_PREVISTA_FIM_REQUISITOS", "DAT_REAL_FIM_REQ", "DIAS_ATRASO_REQUISITOS", "DAT_PREVISTA_INI_DESENV", "DAT_REAL_INI_DESENV", "DAT_PREVISTA_FIM_DESENV", "DAT_REAL_FIM_DESENV","DIAS_ATRASO_DESENVOLVIMENTO", "DAT_PREVISTA_FIM_DEMANDA", "CLASSIFICACAO"]
df_original = pd.read_excel(url_dataset, usecols=colunas_desejadas )


# ANALISAR OS DADOS

In [289]:
# ============================================================
# VERIFICAR O BALANCEAMENTO DA COLUNA TARGET
# ============================================================

# Lista as 5 primeiras linhas do dataframe.
df_original.head()

# target original, antes do pré-processamento
target_inicial = 'CLASSIFICACAO'


balanceamento = df_original[target_inicial].value_counts()
percentual    = df_original[target_inicial].value_counts(normalize=True) * 100

# Exibir resultado
print('Distribuição da variável target:\n')
print(f'🟢 DENTRO DO PRAZO (0): {balanceamento[0]} registros ({percentual[0]:.1f}%)')
print(f'🔴 ATRASO          (1): {balanceamento[1]} registros ({percentual[1]:.1f}%)')

Distribuição da variável target:

🟢 DENTRO DO PRAZO (0): 1470 registros (90.1%)
🔴 ATRASO          (1): 162 registros (9.9%)


# PRÉ-PROCESSAMENTO

In [290]:
# ============================================================
# TÉCNICA: Label Encoding (Codificação de Rótulos)
# ============================================================

# Cria uma cópia do dataframe original para preservá-lo intacto
# Todas as alterações serão feitas apenas no df_ajustado
df_ajustado = df_original.copy()

# Cria uma nova coluna numérica baseada na coluna STATUS_FINAL
# map() substitui cada valor categórico pelo número correspondente
df_ajustado['CLASSIFICACAO_FINAL_NUM'] = df_ajustado['CLASSIFICACAO'].map({
    'ATRASO'         : 1,
    'NO PRAZO': 0
})

# Variavel Target
target = "CLASSIFICACAO_FINAL_NUM"



In [291]:
# ============================================================
# TÉCNICA: One-Hot Encoding
# ============================================================

# Aplicar One-Hot Encoding na coluna SISTEMA
# pd.get_dummies() cria uma coluna binária (0 ou 1) para cada sistema único
# dtype=int garante que os valores sejam inteiros ao invés de booleanos
# Aplicar nas colunas categóricas sem ordem natural
for coluna in ['SISTEMA', 'TIPO_DEMANDA']:
    dummies = pd.get_dummies(df_ajustado[coluna], prefix=coluna, dtype=int)
    df_ajustado = pd.concat([df_ajustado, dummies], axis=1)
    df_ajustado = df_ajustado.drop(columns=[coluna])


In [292]:

# ============================================================
# CONVERTER COLUNAS DE DATA PARA DATETIME
# Necessário para realizar operações matemáticas entre datas
# ============================================================
colunas_data = [
    'DAT_INICIAL_CLASSIFICACAO',
    'DAT_PREVISTA_INI_DEMANDA',
    'DAT_PREVISTA_INI_REQUISITOS',
    'DAT_REAL_INI_REQ',
    'DAT_PREVISTA_FIM_REQUISITOS',
    'DAT_REAL_FIM_REQ',
    'DAT_PREVISTA_INI_DESENV',
    'DAT_REAL_INI_DESENV',
    'DAT_PREVISTA_FIM_DESENV',
    'DAT_REAL_FIM_DESENV',
    'DAT_PREVISTA_FIM_DEMANDA'
]

for coluna in colunas_data:
    df_ajustado[coluna] = pd.to_datetime(
        df_ajustado[coluna], dayfirst=True, errors='coerce'
    )


In [293]:
# ============================================================
# FEATURE ENGINEERING
# ============================================================

# Inclusão de feature para quantidade de dias de atraso no inicio do desenvolvimento.
df_ajustado['ATRASO_INICIO_DESENV']  = (df_ajustado['DAT_REAL_INI_DESENV']      - df_ajustado['DAT_PREVISTA_INI_DESENV']).dt.days

# Atribui -1 quando DIAS_ATRASO_REQUISITOS for nulo
df_ajustado['DIAS_ATRASO_REQUISITOS'] = df_ajustado['DIAS_ATRASO_REQUISITOS'].fillna(-1)


In [294]:
# ============================================================
# DEFINIR AS COLUNAS NECESSÁRIAS PARA PREDIÇÃO
# ============================================================

# Colunas One-Hot Encoding geradas para SISTEMA e TIPO_DEMANDA
colunas_sistema = [col for col in df_ajustado.columns if col.startswith('SISTEMA_')]
colunas_tipo    = [col for col in df_ajustado.columns if col.startswith('TIPO_DEMANDA_')]

# Features numéricas
colunas_numericas = [
    'ATRASO_INICIO_DESENV',
    'DIAS_ATRASO_REQUISITOS'
]

# Concatena todas as colunas necessárias
colunas_modelo = colunas_sistema + colunas_tipo + colunas_numericas + [target]

# Filtra o dataset
df_ajustado = df_ajustado[colunas_modelo]

# Separa features e target
X = df_ajustado.drop(columns=[target])
y = df_ajustado[target]

print(f'Features: {X.shape[1]} colunas')
print(f'Registros: {X.shape[0]}')
print(f'\nColunas utilizadas:\n{X.columns.tolist()}')
print(f'\nDistribuição do target:\n{y.value_counts()}')

Features: 16 colunas
Registros: 1632

Colunas utilizadas:
['SISTEMA_CSA', 'SISTEMA_DFe - Documento de Fiscalização Eletrônico', 'SISTEMA_DPP ', 'SISTEMA_I-SIMP (DPP)', 'SISTEMA_RENOVACALC', 'SISTEMA_SIGAF', 'SISTEMA_SIMP', 'SISTEMA_SRD - GLP', 'SISTEMA_SRD - PR', 'TIPO_DEMANDA_BUG IMPEDITIVO', 'TIPO_DEMANDA_BUG NÃO IMPEDITIVO', 'TIPO_DEMANDA_MELHORIA MÉDIA', 'TIPO_DEMANDA_MELHORIA PEQUENA', 'TIPO_DEMANDA_ORIENTAÇÃO', 'ATRASO_INICIO_DESENV', 'DIAS_ATRASO_REQUISITOS']

Distribuição do target:
CLASSIFICACAO_FINAL_NUM
0    1470
1     162
Name: count, dtype: int64


In [295]:
# ============================================================
# APRESENTA AS COLUNAS DA DATAFRAME
# ============================================================
print('📊 Colunas do dataframe ajustado:')
print(df_ajustado.dtypes)
print(f'\n✅ Shape final: {df_ajustado.shape[0]} linhas x {df_ajustado.shape[1]} colunas')


📊 Colunas do dataframe ajustado:
SISTEMA_CSA                                             int64
SISTEMA_DFe - Documento de Fiscalização Eletrônico      int64
SISTEMA_DPP                                             int64
SISTEMA_I-SIMP (DPP)                                    int64
SISTEMA_RENOVACALC                                      int64
SISTEMA_SIGAF                                           int64
SISTEMA_SIMP                                            int64
SISTEMA_SRD - GLP                                       int64
SISTEMA_SRD - PR                                        int64
TIPO_DEMANDA_BUG IMPEDITIVO                             int64
TIPO_DEMANDA_BUG NÃO IMPEDITIVO                         int64
TIPO_DEMANDA_MELHORIA MÉDIA                             int64
TIPO_DEMANDA_MELHORIA PEQUENA                           int64
TIPO_DEMANDA_ORIENTAÇÃO                                 int64
ATRASO_INICIO_DESENV                                    int64
DIAS_ATRASO_REQUISITOS               

#ANALISE DOS DADOS

#TESTANDO OS MODELOS

In [296]:
# ============================================================
# DIVIDIR DADOS DE TREINO E TESTE
# ============================================================

# ----------------------------------------------------------
# Dividir em treino (80%) e teste (20%)
# stratify=y garante que a proporção de 0 e 1 seja mantida
# igual nos dois conjuntos — essencial para dados desbalanceados
# random_state=42 garante que a divisão seja reproduzível
# ----------------------------------------------------------
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size    = 0.20,
    stratify     = y,
    random_state = 42
)

# Exibir o resultado da divisão
print(f'Total de registros  : {len(X)}')
print(f'Registros de treino : {len(X_treino)} ({len(X_treino)/len(X)*100:.1f}%)')
print(f'Registros de teste  : {len(X_teste)} ({len(X_teste)/len(X)*100:.1f}%)')
print(f'\nDistribuição do target no treino:\n{y_treino.value_counts()}')
print(f'\nDistribuição do target no teste:\n{y_teste.value_counts()}')

Total de registros  : 1632
Registros de treino : 1305 (80.0%)
Registros de teste  : 327 (20.0%)

Distribuição do target no treino:
CLASSIFICACAO_FINAL_NUM
0    1175
1     130
Name: count, dtype: int64

Distribuição do target no teste:
CLASSIFICACAO_FINAL_NUM
0    295
1     32
Name: count, dtype: int64


In [297]:
# ============================================================
# CRIAR DOS SINTETICOS PARA MELHORAR BALANCEAMENTO - TESTE
# ============================================================

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_treino, y_treino)

print(f"Antes do SMOTE : {y_treino.value_counts().to_dict()}")
print(f"Depois do SMOTE: {y_train_bal.value_counts().to_dict()}")

Antes do SMOTE : {0: 1175, 1: 130}
Depois do SMOTE: {0: 1175, 1: 1175}


In [298]:
# ============================================================
# DIVIDIR DADOS DE TREINO E TESTE
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Treino : {X_train.shape[0]} registros')
print(f'Teste  : {X_test.shape[0]} registros')
print(f'\nDistribuição no treino:\n{y_train.value_counts()}')
print(f'\nDistribuição no teste:\n{y_test.value_counts()}')

Treino : 1305 registros
Teste  : 327 registros

Distribuição no treino:
CLASSIFICACAO_FINAL_NUM
0    1175
1     130
Name: count, dtype: int64

Distribuição no teste:
CLASSIFICACAO_FINAL_NUM
0    295
1     32
Name: count, dtype: int64


In [299]:
# ============================================================
# DEFINIR OS HIPERPARÂMETROS DOS MODELOS A SEREM TESTADOS
# ============================================================
modelos = {

    # Baseline simples — sem balanceamento
    'Regressão Logística (sem balanceamento)': (
        LogisticRegression(
            C=100,
            penalty='l2',
            solver='lbfgs',
            max_iter=1000,
            random_state=42
        ),
        'sem_smote', None
    ),

    # Regressão Logística com penalização da classe minoritária
    'Regressão Logística (class_weight)': (
        LogisticRegression(
            C=10,
            penalty='l2',
            solver='lbfgs',
            class_weight='balanced',   # ← restaurado
            max_iter=1000,
            random_state=42
        ),
        'sem_smote', None
    ),

    # Random Forest sem balanceamento
    'Random Forest (sem balanceamento)': (
        RandomForestClassifier(
            max_depth=None,
            min_samples_leaf=1,
            min_samples_split=10,
            n_estimators=100,          # ← corrigido : para =
            random_state=42
        ),
        'sem_smote', None
    ),

    # Random Forest com penalização
    'Random Forest (class_weight)': (
        RandomForestClassifier(
            max_depth=5,               # ← corrigido : para =
            min_samples_leaf=4,
            min_samples_split=2,
            n_estimators=300,
            class_weight='balanced',   # ← restaurado
            random_state=42
        ),
        'sem_smote', None
    ),

    # Random Forest com SMOTE
    'Random Forest (SMOTE)': (
        RandomForestClassifier(n_estimators=200, random_state=42),
        'smote', None
    ),

    # XGBoost sem balanceamento
    'XGBoost (sem balanceamento)': (
        XGBClassifier(
            colsample_bytree=1.0,
            learning_rate=0.05,
            max_depth=6,
            n_estimators=300,
            scale_pos_weight=1,
            subsample=1.0,
            eval_metric='logloss',
            random_state=42
        ),
        'sem_smote', None
    ),

    # XGBoost com scale_pos_weight
    'XGBoost (scale_pos_weight)': (
        XGBClassifier(
            colsample_bytree=0.6,
            learning_rate=0.01,
            max_depth=4,
            n_estimators=200,
            scale_pos_weight=9,
            subsample=1.0,
            eval_metric='logloss',
            random_state=42
        ),
        'sem_smote', None
    ),

    # XGBoost com SMOTE
    'XGBoost (SMOTE)': (
        XGBClassifier(
            colsample_bytree=0.6,
            learning_rate=0.1,
            max_depth=5,
            n_estimators=200,
            subsample=1.0,
            eval_metric='logloss',
            random_state=42
        ),
        'smote', None
    ),
}

In [300]:
# ============================================================
# TREINAR OS MODELOS
# ============================================================

THRESHOLD = 0.20  # threshold definido após análise de negócio

resultados = []

for nome, (modelo, estrategia, _) in modelos.items():

    # Aplica SMOTE apenas no treino quando indicado
    if estrategia == 'smote':
        X_treino_bal, y_treino_bal = SMOTE(random_state=42).fit_resample(X_train, y_train)
    else:
        X_treino_bal, y_treino_bal = X_train, y_train

    # Treina o modelo
    modelo.fit(X_treino_bal, y_treino_bal)

    # Gera probabilidades no conjunto de teste
    y_pred_prob = modelo.predict_proba(X_test)[:, 1]

    # Aplica o threshold definido no lugar do padrão 0.50
    y_pred = (y_pred_prob >= THRESHOLD).astype(int)

    # Métricas focadas na classe minoritária ATRASO
    report = classification_report(y_test, y_pred, output_dict=True)
    auc    = roc_auc_score(y_test, y_pred_prob)

    # Identifica qual label corresponde a ATRASO (1 ou 0 dependendo do LabelEncoder)
    classe_atraso = str(y_test.unique().max())

    resultados.append({
        'Modelo'           : nome,
        'Threshold'        : THRESHOLD,
        'F1 ATRASO'        : round(report[classe_atraso]['f1-score'], 3),
        'Precision ATRASO' : round(report[classe_atraso]['precision'], 3),
        'Recall ATRASO'    : round(report[classe_atraso]['recall'], 3),
        'AUC-ROC'          : round(auc, 3),
        'Acurácia'         : round(report['accuracy'], 3),
    })

    print(f'\n{"="*60}')
    print(f'Modelo: {nome}  |  Threshold: {THRESHOLD}')
    print(f'{"="*60}')
    print(classification_report(y_test, y_pred, target_names=['NO PRAZO', 'ATRASO']))
    print(f'AUC-ROC: {auc:.3f}')
    print(f'Matriz de Confusão:\n{confusion_matrix(y_test, y_pred)}')


Modelo: Regressão Logística (sem balanceamento)  |  Threshold: 0.2
              precision    recall  f1-score   support

    NO PRAZO       0.96      0.97      0.96       295
      ATRASO       0.67      0.62      0.65        32

    accuracy                           0.93       327
   macro avg       0.81      0.80      0.80       327
weighted avg       0.93      0.93      0.93       327

AUC-ROC: 0.886
Matriz de Confusão:
[[285  10]
 [ 12  20]]

Modelo: Regressão Logística (class_weight)  |  Threshold: 0.2
              precision    recall  f1-score   support

    NO PRAZO       0.98      0.55      0.70       295
      ATRASO       0.18      0.91      0.30        32

    accuracy                           0.58       327
   macro avg       0.58      0.73      0.50       327
weighted avg       0.90      0.58      0.66       327

AUC-ROC: 0.882
Matriz de Confusão:
[[161 134]
 [  3  29]]

Modelo: Random Forest (sem balanceamento)  |  Threshold: 0.2
              precision    recall  f1

**Precision** — Dos alertas de ATRASO que o modelo disparou, quantos eram atrasos de verdade.

"Quando o modelo grita atraso, eu posso confiar?"

**Recall** — De todos os atrasos reais que aconteceram, quantos o modelo conseguiu detectar.

"O modelo está deixando atrasos passarem sem alertar?"

**F1** — Média entre Precision e Recall. Útil para comparar modelos quando os dois importam.

"Equilíbrio geral do modelo."

# AVALIAÇÃO E COMPARATIVO

In [301]:
# ============================================================
# APRESENTAR TABELA COMPARATIVA DOS MODELOS TESTADOS
# ============================================================

df_resultados = pd.DataFrame(resultados).sort_values('Recall ATRASO', ascending=False)

print('\n\n══════════════════════════════════════════════════════════════')
print('COMPARATIVO GERAL — ordenado por Recall ATRASO')
print('══════════════════════════════════════════════════════════════')
print(df_resultados.to_string(index=False))



══════════════════════════════════════════════════════════════
COMPARATIVO GERAL — ordenado por Recall ATRASO
══════════════════════════════════════════════════════════════
                                 Modelo  Threshold  F1 ATRASO  Precision ATRASO  Recall ATRASO  AUC-ROC  Acurácia
             XGBoost (scale_pos_weight)        0.2      0.230             0.130          0.969    0.857     0.364
     Regressão Logística (class_weight)        0.2      0.297             0.178          0.906    0.882     0.581
           Random Forest (class_weight)        0.2      0.253             0.147          0.906    0.869     0.477
                        XGBoost (SMOTE)        0.2      0.442             0.333          0.656    0.811     0.838
Regressão Logística (sem balanceamento)        0.2      0.645             0.667          0.625    0.886     0.933
      Random Forest (sem balanceamento)        0.2      0.613             0.633          0.594    0.863     0.927
                  Random Fo

In [302]:
# ============================================================
# PREVER COM NOVOS REGISTROS
# ============================================================


modelo_escolhido = 'Regressão Logística (sem balanceamento)'

THRESHOLD = 0.20

# ── Registros de demandas em andamento para teste ─────────────────────────────
# NUM_DEMANDA e ANO_DEMANDA: apenas identificação, não entram no modelo
# ATRASO_INICIO_DESENV: positivo = atrasou, negativo = adiantado, 0 = no prazo
# DIAS_ATRASO_REQUISITOS: -1 = sem etapa de requisitos, 0 = no prazo, >0 = atrasou

novos_registros = pd.DataFrame([
    {
        'NUM_DEMANDA'            : '528',
        'ANO_DEMANDA'            : 2026,
        'SISTEMA'                : 'SIGAF',
        'TIPO_DEMANDA'           : 'MELHORIA MÉDIA',
        'ATRASO_INICIO_DESENV'   : 5,
        'DIAS_ATRASO_REQUISITOS' : 3,
    },
    {
        'NUM_DEMANDA'            : '58',
        'ANO_DEMANDA'            : 2026,
        'SISTEMA'                : 'SIMP',
        'TIPO_DEMANDA'           : 'BUG IMPEDITIVO',
        'ATRASO_INICIO_DESENV'   : 0,
        'DIAS_ATRASO_REQUISITOS' : -1,
    },
    {
        'NUM_DEMANDA'            : '150',
        'ANO_DEMANDA'            : 2026,
        'SISTEMA'                : 'CSA',
        'TIPO_DEMANDA'           : 'MELHORIA PEQUENA',
        'ATRASO_INICIO_DESENV'   : 2,
        'DIAS_ATRASO_REQUISITOS' : -1,
    },
    {
        'NUM_DEMANDA'            : '99',
        'ANO_DEMANDA'            : 2026,
        'SISTEMA'                : 'RENOVACALC',
        'TIPO_DEMANDA'           : 'MELHORIA MÉDIA',
        'ATRASO_INICIO_DESENV'   : 8,
        'DIAS_ATRASO_REQUISITOS' : 6,
    },
    {
        'NUM_DEMANDA'            : '202',
        'ANO_DEMANDA'            : 2026,
        'SISTEMA'                : 'SRD - GLP',
        'TIPO_DEMANDA'           : 'BUG NÃO IMPEDITIVO',
        'ATRASO_INICIO_DESENV'   : 1,
        'DIAS_ATRASO_REQUISITOS' : -1,
    },
])

# ── Separa identificadores antes do processamento ────────────────────────────
# NUM_DEMANDA e ANO_DEMANDA são removidos aqui e reincluídos apenas no resultado
colunas_id      = ['NUM_DEMANDA', 'ANO_DEMANDA']
colunas_modelo  = ['SISTEMA', 'TIPO_DEMANDA', 'ATRASO_INICIO_DESENV', 'DIAS_ATRASO_REQUISITOS']

novos_para_modelo = novos_registros[colunas_modelo].copy()

# ── Aplicar o mesmo One-Hot Encoding do treino ────────────────────────────────
novos_dummies   = pd.get_dummies(novos_para_modelo, columns=['SISTEMA', 'TIPO_DEMANDA'])

# Alinha as colunas com as do modelo — preenche com 0 colunas ausentes
novos_alinhados = novos_dummies.reindex(columns=X.columns, fill_value=0)

# ── Gerar probabilidades e classificação ──────────────────────────────────────
modelo_lr, _, _ = modelos[modelo_escolhido]
modelo_lr.fit(X_train, y_train)

probabilidades = modelo_lr.predict_proba(novos_alinhados)[:, 1]
classificacoes = ['ATRASO' if p >= THRESHOLD else 'NO PRAZO' for p in probabilidades]

# ── Resultado final — identificadores reincluídos apenas na exibição ──────────
resultado = novos_registros.copy()
resultado['PROB_ATRASO']   = [f'{p:.1%}' for p in probabilidades]
resultado['CLASSIFICACAO'] = classificacoes
resultado['RISCO DE ATRASO'] = [
    '🔴 ALTO'  if p >= 0.50       else
    '🟡 MÉDIO' if p >= THRESHOLD  else
    '🟢 BAIXO'
    for p in probabilidades
]

print(resultado[[
    'NUM_DEMANDA',
    'ANO_DEMANDA',
    'SISTEMA',
    'TIPO_DEMANDA',
    'DIAS_ATRASO_REQUISITOS',
    'ATRASO_INICIO_DESENV',
    'RISCO DE ATRASO'
]].to_string(index=False))

NUM_DEMANDA  ANO_DEMANDA    SISTEMA       TIPO_DEMANDA  DIAS_ATRASO_REQUISITOS  ATRASO_INICIO_DESENV RISCO DE ATRASO
        528         2026      SIGAF     MELHORIA MÉDIA                       3                     5          🔴 ALTO
         58         2026       SIMP     BUG IMPEDITIVO                      -1                     0         🟢 BAIXO
        150         2026        CSA   MELHORIA PEQUENA                      -1                     2         🟢 BAIXO
         99         2026 RENOVACALC     MELHORIA MÉDIA                       6                     8          🔴 ALTO
        202         2026  SRD - GLP BUG NÃO IMPEDITIVO                      -1                     1         🟢 BAIXO


# OTIMIZAÇÃO

In [303]:
# ============================================================
# TESTAR OUTROS VALORES DE THRESHOLD PARA RANDOM FOREST - 1
# ============================================================

modelo_rf = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_rf.fit(X_train, y_train)
y_prob = modelo_rf.predict_proba(X_test)[:, 1]

print(f'{"Threshold":<12} {"F1":<8} {"Precision":<12} {"Recall":<10} {"AUC-ROC"}')
print('-' * 55)

for threshold in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]:
    y_pred_t = (y_prob >= threshold).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    classe   = str(y_test.unique().max())
    print(f'{threshold:<12} '
          f'{report[classe]["f1-score"]:<8.3f} '
          f'{report[classe]["precision"]:<12.3f} '
          f'{report[classe]["recall"]:<10.3f} '
          f'{roc_auc_score(y_test, y_prob):.3f}')

Threshold    F1       Precision    Recall     AUC-ROC
-------------------------------------------------------
0.5          0.625    0.938        0.469      0.821
0.45         0.612    0.882        0.469      0.821
0.4          0.600    0.833        0.469      0.821
0.35         0.577    0.750        0.469      0.821
0.3          0.566    0.714        0.469      0.821
0.25         0.586    0.654        0.531      0.821
0.2          0.610    0.667        0.562      0.821


In [304]:
# ============================================================
# TESTAR OUTROS VALORES DE THRESHOLD PARA RANDOM FOREST - 2
# ============================================================

for threshold in [0.20, 0.15, 0.10, 0.08, 0.05]:
    y_pred_t = (y_prob >= threshold).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    classe   = str(y_test.unique().max())
    print(f'Threshold {threshold} → '
          f'Precision={report[classe]["precision"]:.3f} | '
          f'Recall={report[classe]["recall"]:.3f} | '
          f'F1={report[classe]["f1-score"]:.3f}')

Threshold 0.2 → Precision=0.667 | Recall=0.562 | F1=0.610
Threshold 0.15 → Precision=0.413 | Recall=0.594 | F1=0.487
Threshold 0.1 → Precision=0.400 | Recall=0.625 | F1=0.488
Threshold 0.08 → Precision=0.370 | Recall=0.625 | F1=0.465
Threshold 0.05 → Precision=0.278 | Recall=0.688 | F1=0.396


In [ ]:
# ============================================================
# AVALIAR HIPERPARÂMETROS DOS MODELOS RANDOM FOREST E REGRESSÃO LOGISTICA
# ============================================================

# scorer baseado no treino — otimiza Recall da classe ATRASO
scorer_recall = make_scorer(recall_score, pos_label=y_train.unique().max())

# ── Grids de hiperparâmetros por modelo ───────────────────────────────────────
param_grid_rf = {
    'n_estimators'     : [100, 200, 300, 500],
    'max_depth'        : [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4],
}

param_grid_lr = {
    'C'      : [0.001, 0.01, 0.1, 1, 10, 100],
    'solver' : ['lbfgs', 'saga'],
    'penalty': ['l2'],
}

# ── Definição dos modelos a otimizar ──────────────────────────────────────────
modelos_grid = {

    'Random Forest (sem balanceamento)': (
        RandomForestClassifier(random_state=42),
        param_grid_rf,
        'sem_smote'
    ),

    'Random Forest (class_weight)': (
        RandomForestClassifier(class_weight='balanced', random_state=42),
        param_grid_rf,
        'sem_smote'
    ),

    'Regressão Logística (sem balanceamento)': (
        LogisticRegression(max_iter=1000, random_state=42),
        param_grid_lr,
        'sem_smote'
    ),

    'Regressão Logística (class_weight)': (
        LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        param_grid_lr,
        'sem_smote'
    ),
}

# ── GridSearch para cada modelo ───────────────────────────────────────────────
resultados_grid = {}

for nome, (modelo_base, grid, estrategia) in modelos_grid.items():

    print(f'\n{"="*60}')
    print(f'Otimizando: {nome}')
    print(f'{"="*60}')

    # todos sem SMOTE — apenas dados reais
    X_tr, y_tr = X_train, y_train

    gs = GridSearchCV(
        modelo_base,
        grid,
        scoring=scorer_recall,
        cv=5,
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_tr, y_tr)

    # avalia com threshold 0.20
    y_prob = gs.best_estimator_.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.20).astype(int)

    report = classification_report(y_test, y_pred, output_dict=True)
    classe = str(y_train.unique().max())
    auc    = roc_auc_score(y_test, y_prob)

    resultados_grid[nome] = {
        'Melhores parâmetros': gs.best_params_,
        'Recall'             : round(report[classe]['recall'],    3),
        'Precision'          : round(report[classe]['precision'], 3),
        'F1'                 : round(report[classe]['f1-score'],  3),
        'AUC-ROC'            : round(auc, 3),
        'Acurácia'           : round(report['accuracy'], 3),
    }

    print(f'Melhores parâmetros : {gs.best_params_}')
    print(f'Recall no treino (cv): {gs.best_score_:.3f}')
    print(f'\n{classification_report(y_test, y_pred, target_names=["NO PRAZO", "ATRASO"])}')
    print(f'AUC-ROC : {auc:.3f}')
    print(f'Matriz de Confusão:\n{confusion_matrix(y_test, y_pred)}')

# ── Comparativo final ─────────────────────────────────────────────────────────
print('\n\n' + '═'*62)
print('COMPARATIVO FINAL — ordenado por Recall ATRASO')
print('═'*62)

df_resultado = pd.DataFrame({
    nome: {k: v for k, v in vals.items() if k != 'Melhores parâmetros'}
    for nome, vals in resultados_grid.items()
}).T.sort_values('Recall', ascending=False)

print(df_resultado.to_string())

# ── Melhores parâmetros encontrados ──────────────────────────────────────────
print('\n\n' + '═'*62)
print('MELHORES PARÂMETROS ENCONTRADOS')
print('═'*62)
for nome, vals in resultados_grid.items():
    print(f'\n{nome}')
    print(f'  → {vals["Melhores parâmetros"]}')



Otimizando: Random Forest (sem balanceamento)


In [ ]:
# ============================================================
# AVALIAR HIPERPARÂMETROS DOS MODELOS XGBOOST
# ============================================================


# scorer baseado no treino
scorer_recall = make_scorer(recall_score, pos_label=y_train.unique().max())

param_grid = {
    'n_estimators'    : [100, 200, 300, 500],
    'max_depth'       : [3, 4, 5, 6],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.2],
    'subsample'       : [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

resultados_grid = {}

for nome, (modelo_base, estrategia) in {
    'XGBoost (sem balanceamento)': (XGBClassifier(eval_metric='logloss', random_state=42), 'sem_smote'),
    'XGBoost (scale_pos_weight)' : (XGBClassifier(eval_metric='logloss', random_state=42), 'sem_smote'),
    'XGBoost (SMOTE)'            : (XGBClassifier(eval_metric='logloss', random_state=42), 'smote'),
}.items():

    # Prepara os dados conforme a estratégia
    if estrategia == 'smote':
        X_tr, y_tr = SMOTE(random_state=42).fit_resample(X_train, y_train)
    else:
        X_tr, y_tr = X_train, y_train

    # Grid específico por versão
    if nome == 'XGBoost (sem balanceamento)':
        grid = {**param_grid, 'scale_pos_weight': [1]}
    elif nome == 'XGBoost (scale_pos_weight)':
        grid = {**param_grid, 'scale_pos_weight': [1, 3, 5, 9]}
    else:
        grid = param_grid  # SMOTE: scale_pos_weight removido do base

    # GridSearch otimizando Recall
    gs = GridSearchCV(
        modelo_base, grid,
        scoring=scorer_recall,
        cv=5, n_jobs=-1, verbose=0
    )
    gs.fit(X_tr, y_tr)

    # Avalia com threshold 0.20
    y_prob = gs.best_estimator_.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.20).astype(int)

    report = classification_report(y_test, y_pred, output_dict=True)
    classe = str(y_train.unique().max())

    resultados_grid[nome] = {
        'melhores_params': gs.best_params_,
        'Recall'         : round(report[classe]['recall'], 3),
        'Precision'      : round(report[classe]['precision'], 3),
        'F1'             : round(report[classe]['f1-score'], 3),
        'AUC-ROC'        : round(roc_auc_score(y_test, y_prob), 3)
    }

    print(f'\n{"="*55}')
    print(f'{nome}')
    print(f'Melhores parâmetros: {gs.best_params_}')
    print(classification_report(y_test, y_pred, target_names=['NO PRAZO', 'ATRASO']))

# Comparativo final
df_grid = pd.DataFrame(resultados_grid).T
print('\nCOMPARATIVO FINAL')
print(df_grid[['Recall','Precision','F1','AUC-ROC']].sort_values('Recall', ascending=False))